In [0]:
from pyspark.sql import DataFrame, functions as F
from datetime import datetime, timezone


def log_pipeline_run(spark, platform, layer, start_dt, end_dt):
    duration = round((end_dt - start_dt).total_seconds(), 2)
    log_row = spark.createDataFrame([{
        "platform": platform, "layer": layer,
        "start_ts": start_dt, "end_ts": end_dt, "duration_seconds": duration,
    }])
    log_row.write.format("delta").mode("append").saveAsTable("pipeline_run_log")
    print(f"[{platform}/{layer}] duration: {duration}s")

In [0]:
Z_PREFIX_COLS = ["MSTATUS", "GENDER", "EDUCATION", "OCCUPATION", "CAR_TYPE", "URBANICITY"]
NUMERIC_IMPUTE_COLS = ["AGE", "YOJ", "INCOME", "HOME_VAL", "CAR_AGE"]
FROZEN_MEDIANS = {"AGE": 45.0, "YOJ": 11.0, "INCOME": 53758.0, "HOME_VAL": 160485.0, "CAR_AGE": 8.0}

In [0]:
def clean_policyholders_silver(df: DataFrame, medians: dict) -> DataFrame:
    for c in Z_PREFIX_COLS:
        df = df.withColumn(c, F.regexp_replace(F.col(c), r"^z_", ""))
    df = df.withColumn("RED_CAR", F.initcap(F.col("RED_CAR")))
    df = df.withColumn("BIRTH_DATE", F.to_date(F.col("BIRTH"), "ddMMMyy"))
    df = df.withColumn("CAR_AGE", F.when(F.col("CAR_AGE") < 0, None).otherwise(F.col("CAR_AGE")))
    for c in NUMERIC_IMPUTE_COLS:
        df = df.withColumn(f"{c}_IMPUTED", F.col(c).isNull())
        df = df.withColumn(c, F.when(F.col(c).isNull(), medians[c]).otherwise(F.col(c)))
    df = df.withColumn("OCCUPATION", F.when(F.col("OCCUPATION").isNull(), "Unknown").otherwise(F.col("OCCUPATION")))
    return df

In [0]:
def clean_telematics_silver(df: DataFrame) -> DataFrame:
    df = df.dropDuplicates(["device_id", "timestamp", "PID", "value", "alarm_class"])
    df = df.withColumn(
        "alarm_class",
        F.when((F.col("alarm_class") >= 0) & (F.col("alarm_class") <= 4), F.col("alarm_class")).otherwise(None)
    )
    pid_stats = df.groupBy("PID").agg(F.mean("value").alias("pid_mean"), F.stddev("value").alias("pid_std"))
    pid_stats = pid_stats.fillna({"pid_std": 0.01})
    df = df.join(pid_stats, on="PID", how="left")
    df = df.withColumn("VALUE_OUTLIER", F.abs(F.col("value") - F.col("pid_mean")) > (4 * F.col("pid_std")))
    df = df.drop("pid_mean", "pid_std")
    return df

In [0]:
start_dt = datetime.now(timezone.utc)

bronze_policyholders = spark.read.table("bronze_car_insurance_claim")
bronze_telematics = spark.read.table("bronze_telematics_events")

silver_policyholders = clean_policyholders_silver(bronze_policyholders, FROZEN_MEDIANS)
silver_policyholders.write.format("delta").mode("overwrite").saveAsTable("silver_car_insurance_claim")

silver_telematics = clean_telematics_silver(bronze_telematics)
silver_telematics.write.format("delta").mode("overwrite").saveAsTable("silver_telematics_events")

end_dt = datetime.now(timezone.utc)
log_pipeline_run(spark, "Databricks", "silver", start_dt, end_dt)

In [0]:
silver_policyholders_check = spark.read.table("silver_car_insurance_claim")
silver_telematics_check = spark.read.table("silver_telematics_events")
print("silver_car_insurance_claim rows:", silver_policyholders_check.count())
print("silver_telematics_events rows:", silver_telematics_check.count())